# Folio: one PDF, saved work, verifiable answers

Run the cells in order using the project's Python kernel. Every HTTP request is visible below. The small `checkpoint.py` helper saves request intent and output files; it does not implement Folio processing.

The example uses the EPA's [Home Buyer's and Seller's Guide to Radon](https://www.epa.gov/sites/default/files/2015-05/documents/hmbuygud.pdf). Download it outside tracked source, then set `PDF_PATH`. The two questions concern this document, not an assessment of a particular home or current professional guidance.

**Upload schedules extraction, then automatic summary and indexing.** Reading a later cell does not trigger those stages; workers may finish while you inspect earlier output. Summary and index progress are independent. Chat requires a ready index.

Keep one kernel using each run directory. Rerunning a submission cell reuses its saved key and exact body. After a kernel restart, rerun from setup with the same directory. A timeout leaves accepted work running. Missing saved server IDs stop the walkthrough; they do not trigger replacement uploads. Do not delete checkpoints to recover from a network error. Database resets before a first receipt is saved cannot be reliably detected by this client.

Use the existing local stack and agreed AI allowance. Upload can incur summary/embedding charges, and questions can incur model charges. This notebook does not enable stages, change settings, reset data or call providers directly.

The notebook is a second client of Folio, like the browser. Python runs each code cell in a persistent kernel, so later cells can use earlier variables. Restarting the kernel loses those variables; the JSON checkpoints on disk let the next execution reconnect to accepted work.

```text
Notebook or browser -> HTTP API -> PostgreSQL: accepted work and results
                               -> artifact volume: original PDF and raw extraction
Dispatcher -> Redis/RQ -> worker -> parser or model -> PostgreSQL: saved outcome
Notebook or browser <- HTTP GET <- API reads that saved outcome
```

Follow one question throughout: does an electret ion chamber detector need electricity, and can it test for more than 90 days? The answer needs two passages. Watching both reach the model is more informative than checking whether a plausible answer appears.


In [ ]:
import hashlib
import json
import os
import subprocess
import sys
import time
from datetime import UTC, datetime, timedelta
from pathlib import Path
from urllib.parse import urlsplit

import httpx

REPO = next(
    folder
    for folder in (Path.cwd(), *Path.cwd().parents)
    if (folder / "pyproject.toml").is_file() and (folder / "src" / "folio").is_dir()
)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
from notebooks.checkpoint import bind_run, save_json, submit_once  # noqa: E402

API_URL = "http://127.0.0.1:8000"
PDF_PATH = REPO / "output/retrieval-comparison/source/epa-radon-home-buyers-sellers-2024.pdf"
RUN_DIR = REPO / "output/demo/folio"
WAIT_SECONDS = 600
POLL_SECONDS = 2

# Optional log collection uses the current Docker context unless explicitly set here.
SHOW_FULL_INSPECTION = False
CAPTURE_WORKER_LOGS = False
DOCKER_HOST = None

parsed = urlsplit(API_URL)
if (
    parsed.scheme not in {"http", "https"}
    or parsed.hostname not in {"localhost", "127.0.0.1", "::1"}
    or parsed.username is not None
    or parsed.password is not None
    or parsed.path not in {"", "/"}
    or parsed.query
    or parsed.fragment
):
    raise ValueError("Use a local API origin without credentials, query or path.")
API_URL = API_URL.rstrip("/")
RUN_DIR = RUN_DIR.resolve()
if not RUN_DIR.is_relative_to((REPO / "output").resolve()):
    raise ValueError("Keep generated results under the ignored output/ directory.")
pdf_bytes = PDF_PATH.read_bytes()
identity = {
    "api_url": API_URL,
    "filename": PDF_PATH.name,
    "bytes": len(pdf_bytes),
    "sha256": hashlib.sha256(pdf_bytes).hexdigest(),
}
run = bind_run(RUN_DIR / "run.json", identity)


last_http = None


def show_http(response):
    global last_http
    call = (response.request.method, response.request.url.path, response.status_code)
    if call != last_http:
        print(call[0], call[1], "->", call[2])
    last_http = call
    response.raise_for_status()


if "client" in globals():
    globals()["client"].close()
client = httpx.Client(
    base_url=API_URL,
    timeout=30,
    trust_env=False,
    follow_redirects=False,
    event_hooks={"response": [show_http]},
)
print("Run:", RUN_DIR)
print("Source:", identity)

## 1. Read the running application

Health checks API/database access. It does not establish worker or provider health. Development settings expose the existing shared allowance, not provider keys. Baseline metrics remain the original snapshot when this cell is rerun.


In [ ]:
health = client.get("/health").json()
capabilities = client.get("/capabilities").json()
save_json(RUN_DIR / "capabilities.json", capabilities)
if not (RUN_DIR / "upload.json").exists() and len(pdf_bytes) > capabilities["max_upload_bytes"]:
    raise ValueError("This PDF exceeds the current upload limit.")
if capabilities["development_mode"]:
    settings = client.get("/development/settings").json()
    if not (RUN_DIR / "settings-before.json").exists():
        save_json(RUN_DIR / "settings-before.json", settings)
    print("Saved summary enablement:", settings["values"]["summary_enabled"])
    print("Existing allowance:", settings["runtime"])
else:
    print("Development inspection is disabled; public status remains available.")

baseline_path = RUN_DIR / "metrics-before.json"
if not baseline_path.exists():
    save_json(
        baseline_path,
        {
            "documents": client.get("/statistics/documents").json(),
            "processing": client.get("/metrics/processing").json(),
        },
    )
print("Health:", health)
display(json.loads(baseline_path.read_text(encoding="utf-8")))

## 2. Accept the original PDF

The helper saves the filename/hash and idempotency key before this POST. If its response is lost, rerun this cell: the same bytes, filename and key recover the existing receipt. A saved receipt causes a status GET, not another upload. The receipt means acceptance, not successful extraction.

The API stores the original file, then commits the document, request and initial attempt together in a database transaction. A transaction makes those database changes succeed or fail together. File storage and Redis are separate systems, so the database also retains work that the dispatcher can deliver later. An API crash after acceptance need not lose the job.

The idempotency key names one intended upload. The helper writes it to disk before sending. A network timeout might mean the server accepted the upload but its reply was lost; reusing the key prevents that uncertainty from becoming a duplicate document. Follow the real path through [acceptance](../src/folio/ingestion/acceptance.py) and [checkpoint.py](checkpoint.py).


In [ ]:
upload = submit_once(
    RUN_DIR / "upload.json",
    identity=identity,
    prepare=lambda: {"filename": identity["filename"], "sha256": identity["sha256"]},
    send=lambda body, key: client.post(
        "/documents",
        headers={"Idempotency-Key": key},
        files={"file": (body["filename"], pdf_bytes, "application/pdf")},
        timeout=120,
    ),
    read_saved=lambda saved: client.get(f"/ingestion-requests/{saved['request_id']}"),
)
request_id = upload["request_id"]
document_id = upload["document_id"]
display(upload)

## 3. Observe extraction

An ingestion request stays `open` while its current attempt is pending or running. Retries create attempts inside the same request. The authorized attempt is the one allowed to publish evidence. A delayed old worker cannot replace a newer result.

The timeout below stops this wait, not the server job. Rerun this read cell to continue.

A request is the logical work, such as "extract this PDF". An attempt is one execution of that work. The worker claims an attempt for a limited period, called a lease. Ownership checks prevent a late worker from publishing after recovery has assigned a new attempt. Redis delivers a notification; PostgreSQL decides which execution owns the result. Multiple workers can consume the queue without each being assigned permanently to one user.

Docling runs locally in the document worker. Its completion creates an immutable extraction revision: a saved interpretation of this exact PDF with parser settings and source locations. See [processing](../src/folio/ingestion/processing.py) and [publication](../src/folio/ingestion/publication.py).


In [ ]:
deadline = time.monotonic() + WAIT_SECONDS
previous_progress = None
while True:
    extraction = client.get(f"/ingestion-requests/{request_id}").json()
    if (
        extraction["request_id"] != request_id
        or extraction["document_id"] != document_id
        or extraction["file_version_id"] != upload["file_version_id"]
    ):
        raise ValueError("The server returned a different source identity.")
    save_json(RUN_DIR / "extraction.json", extraction)
    progress = {
        "state": extraction["state"],
        "authorized_attempt_id": extraction["authorized_attempt_id"],
        "attempts": [
            {key: attempt[key] for key in ("id", "attempt_no", "state", "error_code")}
            for attempt in extraction["attempts"]
        ],
    }
    if progress != previous_progress:
        display(progress)
        previous_progress = progress
    if extraction["state"] != "open":
        break
    if time.monotonic() >= deadline:
        raise TimeoutError("Extraction is still open. Rerun this read cell to resume.")
    time.sleep(POLL_SECONDS)
if extraction["state"] != "succeeded":
    raise RuntimeError("Extraction failed; inspect extraction.json before continuing.")
revision_id = extraction["revision_id"]

## 4. Read all source evidence

Evidence is paginated. Each element keeps its physical page locations; table structure and known gaps remain in the saved response. `no_content_recovered` does not prove a blank page. Valid source identifiers do not establish that an answer is supported.

A source element is a recovered paragraph, heading or table, with an identifier and location. It is not yet a retrieval chunk. For example, the detector's no-power statement and the definition of a long-term test occur on different physical pages. Their separate locations survive later grouping, so an answer can point back to both originals.


In [ ]:
document = client.get(f"/documents/{document_id}").json()
if document["current_request_id"] != request_id or document["active_revision_id"] != revision_id:
    raise ValueError("The document no longer points to this run's evidence.")
save_json(RUN_DIR / "document.json", document)

evidence_pages = []
after_order = -1
while True:
    page = client.get(
        f"/revisions/{revision_id}/evidence",
        params={"after_order": after_order, "limit": 100},
    ).json()
    if page["revision_id"] != revision_id or page["request_id"] != request_id:
        raise ValueError("Evidence belongs to a different revision.")
    evidence_pages.append(page)
    next_order = page["next_after_order"]
    if next_order is None:
        break
    if next_order <= after_order:
        raise RuntimeError("Evidence cursor did not advance.")
    after_order = next_order
save_json(RUN_DIR / "evidence.json", evidence_pages)
elements = [element for page in evidence_pages for element in page["elements"]]
print("Revision:", revision_id, "| elements:", len(elements))
display(evidence_pages[0]["pages"])
display(elements[:2])
print("Original PDF:", f"{API_URL}/documents/{document_id}/original")

## 5. Read the automatic summary

This GET does not generate a summary. `not_scheduled`, `unavailable` and `failed` are visible outcomes, and do not block the next index cell. Disabled summary settings are frozen at upload; changing settings later does not backfill this run.

Summary and indexing are separate jobs created from the extraction. The summary worker prepares all usable evidence for one model call, checks its input size, and saves the structured result and citations. It does not use the nearest six chunks: there is no question to search for yet. If the full input does not fit, it reports the limit instead of summarizing only the first pages.


In [ ]:
deadline = time.monotonic() + WAIT_SECONDS
previous_progress = None
while True:
    summary = client.get(f"/revisions/{revision_id}/summary").json()
    save_json(RUN_DIR / "summary.json", summary)
    progress = summary["state"]
    if progress != previous_progress:
        print("Summary:", progress)
        previous_progress = progress
    if summary["state"] not in {"pending", "running"}:
        break
    if time.monotonic() >= deadline:
        print("Summary still running; rerun this cell later. Index inspection can continue.")
        break
    time.sleep(POLL_SECONDS)
display(
    {
        "run_id": summary["run_id"],
        "overview": summary["overview"],
        "points": summary["points"],
        "citations": summary["citations"],
        "coverage": summary["coverage"],
        "usage": summary["usage"],
        "error": summary["error"],
    }
)

## 6. Observe the complete index and read its chunks

Indexing was scheduled by successful extraction. Only a complete index is ready for chat. There is no rebuild/retry POST here: the existing first-index API does not restart an unavailable or failed index.

A chunk links to whole source elements and physical pages. Those links are not verified character highlights. An embedding batch can add a successful attempt without being a retry.

Docling groups evidence into chunks with headings. An embedding model turns each chunk into 1,536 numbers that help compare semantic similarity. PostgreSQL stores those vectors through pgvector, alongside the exact text and source links. The text remains what a person can inspect; similarity is a ranking signal, not proof of an answer.

Batches make progress durable. If batch two fails after batch one succeeded, recovery can reuse the saved first batch. The index becomes ready only when every planned chunk has an embedding, so chat never searches a partly built index. Read [chunk preparation](../src/folio/indexing/chunking.py), then [index processing](../src/folio/indexing/processing.py).


In [ ]:
deadline = time.monotonic() + WAIT_SECONDS
previous_progress = None
while True:
    index = client.get(f"/revisions/{revision_id}/index").json()
    save_json(RUN_DIR / "index.json", index)
    progress = (index["state"], index["embedded_chunks"], index["chunk_count"])
    if progress != previous_progress:
        print("Index state, embedded, planned:", progress)
        previous_progress = progress
    if index["state"] not in {"pending", "running"}:
        break
    if time.monotonic() >= deadline:
        raise TimeoutError("Index is still running. Rerun this read cell to resume.")
    time.sleep(POLL_SECONDS)
if index["state"] != "ready":
    raise RuntimeError(f"Chat needs a ready index. Inspect index.json: {index['error_code']}")
index_id = index["index_id"]

chunk_pages = []
after = 0
while True:
    page = client.get(
        f"/revisions/{revision_id}/index/chunks",
        params={"after": after, "limit": 100},
    ).json()
    if page["index_id"] != index_id or page["revision_id"] != revision_id:
        raise ValueError("Chunks belong to a different index.")
    chunk_pages.append(page)
    next_after = page["next_after"]
    if next_after is None:
        break
    if next_after <= after:
        raise RuntimeError("Chunk cursor did not advance.")
    after = next_after
save_json(RUN_DIR / "chunks.json", chunk_pages)
chunks = [chunk for page in chunk_pages for chunk in page["chunks"]]
print("Complete index:", index_id, "| chunks:", len(chunks))
display(chunks[:2])

## 7. Create or resume one saved conversation

A conversation pins one immutable ready index. Creating it does not call a model. The following two questions use this same conversation.

Pinning the index keeps the conversation's source stable. It prevents an unrelated upload or a future re-extraction from silently changing the evidence behind earlier answers. The document, extraction revision, index, conversation and turn therefore have different IDs: each identifies something with a different lifetime.


In [ ]:
conversation = submit_once(
    RUN_DIR / "conversation.json",
    identity={"api_url": API_URL, "index_id": index_id},
    prepare=lambda: {"index_id": index_id},
    send=lambda body, key: client.post(
        "/conversations",
        json=body,
        headers={"Idempotency-Key": key},
    ),
    read_saved=lambda saved: client.get(f"/conversations/{saved['id']}"),
)
if (
    conversation["document_id"] != document_id
    or conversation["revision_id"] != revision_id
    or conversation["index_id"] != index_id
):
    raise ValueError("Conversation scope differs from this run.")
conversation_id = conversation["id"]
display(conversation)

## 8. Submit the detector question

`question-01` names this particular intent. Its question and `observed_head` are persisted before sending. The head is the sequence number this request follows. If the response is lost, rerunning uses that saved head even if the server has already accepted the turn.

A different question needs a new intent name. Editing a question under an existing name raises an error rather than silently adding paid work. A 409 response means the current conversation needs review; the notebook does not invent a new key or rewrite the head.

The head check is optimistic concurrency control: the client says which last turn it saw, and the API rejects a conflicting order. It does not hold a database lock while waiting for the model.

The chat worker embeds this question, searches only the pinned index, selects up to six whole chunks and builds the answer request. That request includes instructions, the question, available history, retrieved evidence and the response schema. These all consume the model's input window. An output allowance leaves room for its reply. The spending allowance is separate: money is reserved before a paid call and settled when its usage is known.

[Retrieval](../src/folio/conversations/retrieval.py) chooses the passages; [preparation](../src/folio/conversations/preparation.py) formats and validates the request/answer; [processing](../src/folio/conversations/processing.py) coordinates the stages. This separation lets us compare retrieval without rewriting billing or conversation persistence.


In [ ]:
QUESTION_INTENT = "question-01"
QUESTION = (
    "The guide mentions electret ion chamber detectors. Do they need electricity, "
    "and can they be used for a test lasting more than 90 days?"
)
first_receipt = submit_once(
    RUN_DIR / f"{QUESTION_INTENT}.json",
    identity={"api_url": API_URL, "conversation_id": conversation_id, "question": QUESTION},
    prepare=lambda: {
        "question": QUESTION,
        "observed_head": client.get(f"/conversations/{conversation_id}").json()["head_sequence"],
    },
    send=lambda body, key: client.post(
        f"/conversations/{conversation_id}/turns",
        json=body,
        headers={"Idempotency-Key": key},
    ),
    read_saved=lambda saved: client.get(
        f"/conversations/{conversation_id}/turns/{saved['id']}",
    ),
)
if first_receipt["conversation_id"] != conversation_id:
    raise ValueError("Question belongs to a different conversation.")
first_turn_id = first_receipt["id"]
display(first_receipt)

In [ ]:
deadline = time.monotonic() + WAIT_SECONDS
previous_progress = None
while True:
    first_turn = client.get(f"/conversations/{conversation_id}/turns/{first_turn_id}").json()
    save_json(RUN_DIR / "answer-01.json", first_turn)
    progress = (first_turn["state"], first_turn["context_state"])
    if progress != previous_progress:
        print("First question state, context:", progress)
        previous_progress = progress
    if first_turn["state"] not in {"pending", "running"}:
        break
    if time.monotonic() >= deadline:
        raise TimeoutError("Answer is still active. Rerun this read cell to resume.")
    time.sleep(POLL_SECONDS)
display(
    {
        "question": first_turn["question"],
        "resolved_question": first_turn["resolved_question"],
        "answer": first_turn["answer"],
        "citations": first_turn["citations"],
        "coverage": first_turn["coverage"],
        "error": first_turn["error"],
    }
)

## 9. Ask the year-round question in the same conversation

This second question is explicit enough to stand alone while continuing the same topic. It demonstrates ordered saved turns, not a stress test of pronoun resolution or history compression. A ready answer can still be a clarification or an honest lack of evidence; inspect its outcome and passages.

For later questions, a small model call interprets the question using saved conversation context before fresh retrieval. It keeps the original wording too. Older exchanges can eventually be summarized into notes while recent exchanges stay unchanged; the full transcript is still stored. Background notes usually prepare the next turn. A question that cannot fit its history waits for bounded preparation instead of silently dropping context. These two questions are too short to exercise that mechanism.


In [ ]:
if first_turn["state"] in {"pending", "running"}:
    raise RuntimeError("Wait for the first turn before submitting another question.")
QUESTION_INTENT = "question-02"
QUESTION = (
    "If I want a better estimate of my home's year-round radon level, how long should "
    "the test run, and can I stop every test kit after 48 hours?"
)
second_receipt = submit_once(
    RUN_DIR / f"{QUESTION_INTENT}.json",
    identity={"api_url": API_URL, "conversation_id": conversation_id, "question": QUESTION},
    prepare=lambda: {
        "question": QUESTION,
        "observed_head": client.get(f"/conversations/{conversation_id}").json()["head_sequence"],
    },
    send=lambda body, key: client.post(
        f"/conversations/{conversation_id}/turns",
        json=body,
        headers={"Idempotency-Key": key},
    ),
    read_saved=lambda saved: client.get(
        f"/conversations/{conversation_id}/turns/{saved['id']}",
    ),
)
if second_receipt["conversation_id"] != conversation_id:
    raise ValueError("Question belongs to a different conversation.")
second_turn_id = second_receipt["id"]
deadline = time.monotonic() + WAIT_SECONDS
previous_progress = None
while True:
    second_turn = client.get(f"/conversations/{conversation_id}/turns/{second_turn_id}").json()
    save_json(RUN_DIR / "answer-02.json", second_turn)
    progress = (second_turn["state"], second_turn["context_state"])
    if progress != previous_progress:
        print("Second question state, context:", progress)
        previous_progress = progress
    if second_turn["state"] not in {"pending", "running"}:
        break
    if time.monotonic() >= deadline:
        raise TimeoutError(
            "Answer is still active. Rerun this cell with the same intent to resume."
        )
    time.sleep(POLL_SECONDS)
display(
    {
        "question": second_turn["question"],
        "resolved_question": second_turn["resolved_question"],
        "answer": second_turn["answer"],
        "citations": second_turn["citations"],
        "coverage": second_turn["coverage"],
        "error": second_turn["error"],
    }
)

## 10. Inspect stored worker activity and evidence support

Development inspection is optional and read-only. Extraction inspection includes delivery job IDs and events. Summary/index attempts and chat invocations expose separate usage and billing. Public `pending` means persisted pending work, not proof that Redis currently has a queued job.

The browser/notebook calls Folio's API. Workers call parsing/provider code and write database state directly; those calls do not appear as notebook HTTP traffic. The saved invocation records explain them.

For one answer statement, follow its source label to a cited chunk, then the chunk's element links to the original physical page. Check whether the text supports the claim. Retrieval rank and valid IDs alone do not establish this.

Set `SHOW_FULL_INSPECTION=True` in setup to expand the full ingestion record. Complete JSON remains saved regardless of display size.

Compare the request's created/finished times with each stage's invocation time. The former includes queue wait; provider-call time does not explain the whole user wait. An answer can also finish technically and fail validation. That still consumed a model call, so the usage record must survive even when no answer is published.


In [ ]:
inspection = None
turn_inspections = []
memory_inspections = []
if capabilities["development_mode"]:
    inspection = client.get(
        f"/development/ingestion-requests/{request_id}", params={"limit": 200}
    ).json()
    events = list(inspection["events"])
    after_id = inspection["next_after_id"]
    while after_id is not None:
        page = client.get(
            f"/development/ingestion-requests/{request_id}",
            params={"after_id": after_id, "limit": 200},
        ).json()
        events.extend(page["events"])
        next_id = page["next_after_id"]
        if next_id is not None and next_id <= after_id:
            raise RuntimeError("Event cursor did not advance.")
        after_id = next_id
    inspection["events"] = events
    inspection["next_after_id"] = None
    if inspection["source"]["sha256"] != identity["sha256"]:
        raise ValueError("Server original differs from the selected PDF.")
    save_json(RUN_DIR / "ingestion-inspection.json", inspection)
    display(
        {
            "request_id": request_id,
            "extraction_attempts": len(inspection.get("attempts", [])),
            "events": len(events),
            "summary_state": (inspection.get("summary") or {}).get("state"),
            "index_state": (inspection.get("indexing") or {}).get("state"),
        }
    )
    if SHOW_FULL_INSPECTION:
        display(inspection)
    if inspection.get("indexing") and inspection["indexing"].get("has_more"):
        print("Index inspection exposes only its latest 50 attempts; this export is incomplete.")

    memory_ids = set()
    for number, turn_id in enumerate((first_turn_id, second_turn_id), start=1):
        detail = client.get(f"/development/conversation-turns/{turn_id}").json()
        turn_inspections.append(detail)
        save_json(RUN_DIR / f"turn-inspection-{number:02}.json", detail)
        display(
            {
                "turn_id": turn_id,
                "evidence": detail["evidence"],
                "invocations": [
                    {
                        key: invocation.get(key)
                        for key in (
                            "id",
                            "attempt_id",
                            "stage",
                            "billing_state",
                            "cost_usd",
                            "reserved_usd",
                        )
                    }
                    for invocation in detail["invocations"]
                ],
            }
        )
        memory = (detail.get("context_snapshot") or {}).get("memory")
        if memory:
            memory_ids.add(memory["id"])
        if detail.get("memory_dependency_id"):
            memory_ids.add(detail["memory_dependency_id"])
    for memory_id in sorted(memory_ids):
        detail = client.get(f"/development/conversation-memories/{memory_id}").json()
        # Ownership tokens are unnecessary in the exported teaching view.
        detail["memory"].pop("claim_token", None)
        for attempt in detail["attempts"]:
            attempt.pop("token", None)
        memory_inspections.append(detail)
    save_json(RUN_DIR / "memory-inspection.json", memory_inspections)
    print("Known referenced memories:", len(memory_inspections))
else:
    print("Skipped development inspection.")
print("Unreferenced background memories have no list endpoint; a short chat may not create any.")

for turn in (first_turn, second_turn):
    for citation in turn["citations"]:
        chunk = next(chunk for chunk in chunks if chunk["ordinal"] == citation["chunk_ordinal"])
        display(
            {
                "turn_id": turn["id"],
                "source_id": citation["source_id"],
                "pages": citation["pages"],
                "text": citation["text"],
                "element_links": chunk["sources"],
            }
        )

## 11. Export the complete transcript and final totals

A history request without a cursor returns only the latest page. The forward cursor below exports every retained visible turn in this conversation.

Aggregate metrics cover all retained work, including other runs. Differences from the baseline are not isolated demo costs or throughput measurements. An attempt duration excludes queue wait and can include delayed detection of a lost worker.


In [ ]:
history = []
after = 0
while True:
    page = client.get(
        f"/conversations/{conversation_id}/turns",
        params={"after": after, "limit": 100},
    ).json()
    history.extend(page["items"])
    next_after = page["next_after"]
    if next_after is None:
        break
    if next_after <= after:
        raise RuntimeError("History cursor did not advance.")
    after = next_after
save_json(RUN_DIR / "conversation-history.json", history)
totals = {
    "documents": client.get("/statistics/documents").json(),
    "processing": client.get("/metrics/processing").json(),
}
save_json(RUN_DIR / "metrics-after.json", totals)
if capabilities["development_mode"]:
    save_json(RUN_DIR / "settings-after.json", client.get("/development/settings").json())
display(totals)
print("Saved turns:", len(history), "| output:", RUN_DIR)

## 12. Optional bounded worker logs

Set `CAPTURE_WORKER_LOGS=True` in setup if local Docker logs are useful. This reads at most the last hour and a bounded tail, with a subprocess timeout. Set `DOCKER_HOST` only when your Docker setup needs an explicit endpoint; no machine-specific endpoint is committed here.

The export keeps only Folio events matching this run's identities, plus the Compose service/container label, component and PID. PID is local to a container and may be reused. Chat/memory logs do not consistently include attempt IDs; stored attempts remain authoritative. Missing/rotated logs or disabled development inspection do not prove that work never happened.

Use a request, attempt or turn ID to join the HTTP response to its events. The service name tells you which kind of worker handled it, and the process ID identifies that process instance inside its container. Logs explain what happened around a failure; saved attempts remain available after a log rotates or a process restarts.


In [ ]:
if CAPTURE_WORKER_LOGS:
    log_ids = {
        request_id,
        document_id,
        revision_id,
        index_id,
        conversation_id,
        first_turn_id,
        second_turn_id,
    }
    if summary.get("run_id"):
        log_ids.add(summary["run_id"])
    log_ids.update(attempt["id"] for attempt in extraction["attempts"])
    for detail in memory_inspections:
        log_ids.add(detail["memory"]["id"])
    since = max(datetime.fromisoformat(run["started_at"]), datetime.now(UTC) - timedelta(hours=1))
    environment = os.environ.copy()
    if DOCKER_HOST:
        environment["DOCKER_HOST"] = DOCKER_HOST
    try:
        result = subprocess.run(
            [
                "docker",
                "compose",
                "logs",
                "--no-color",
                "--timestamps",
                "--since",
                since.isoformat(),
                "--tail",
                "2000",
                "api",
                "dispatcher",
                "worker",
                "chat-worker",
                "memory-worker",
            ],
            cwd=REPO,
            env=environment,
            capture_output=True,
            text=True,
            encoding="utf-8",
            errors="replace",
            timeout=20,
            check=True,
        )
        records = []
        fields = {
            "timestamp",
            "severity",
            "component",
            "pid",
            "event",
            "request_id",
            "attempt_id",
            "document_id",
            "summary_run_id",
            "index_id",
            "turn_id",
            "memory_id",
            "outcome",
            "error_code",
            "transport_job_id",
        }
        for line in result.stdout.splitlines():
            brace = line.find("{")
            if brace < 0:
                continue
            try:
                event = json.loads(line[brace:])
            except json.JSONDecodeError:
                continue
            if not isinstance(event, dict) or not {"event", "component", "pid"} <= event.keys():
                continue
            if not any(str(event.get(key)) in log_ids for key in fields if key.endswith("_id")):
                continue
            records.append(
                {
                    "service_label": line[:brace].strip(),
                    **{key: value for key, value in event.items() if key in fields},
                }
            )
        save_json(RUN_DIR / "worker-logs.json", records)
        print("Matching structured events:", len(records))
        display(records[-10:])
    except (OSError, subprocess.SubprocessError) as error:
        print(
            "Optional Docker logs unavailable:",
            type(error).__name__,
            "| persisted API outputs remain.",
        )
else:
    print("Optional Docker logs skipped.")

## Reading the result

Start with `run.json` and `upload.json`, then `extraction.json` and `evidence.json`. Follow `summary.json`, `index.json` and `chunks.json`, then the conversation and answer files. Development inspection explains attempts, retrieval and usage; metrics summarize the deployment.

In source, read this notebook's upload and question cells, then `notebooks/checkpoint.py:submit_once`. For application behavior, continue through ingestion routes/publication, indexing routes, and conversation routes/processing. The checkpoint tests demonstrate client recovery only; they do not measure model quality.

Keep live notebook copies and generated outputs under ignored `output/`. Clear this tracked notebook's execution counts and outputs before committing. Saved outputs may contain document text and conversation content. Do not add keys, environment dumps or private documents to tracked files.

Owner exercise: inspect one failed or interrupted attempt, explain which ID identifies the logical work and which identifies an execution, then trace one answer citation to its physical PDF page.
